# CBIR Pipeline - Preprocessing Method 2

This pipeline reads images from `data/face`, applies preprocessing method 2, extracts embeddings, and builds a CBIR index.

Output artifacts are saved as `models/cbir_method2_index.npz` and `models/cbir_method2_meta.json`.

No LBPH model is trained in this notebook.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import json

import cv2
import numpy as np
from numpy.typing import NDArray
import face_recognition

ImageArray = NDArray[np.uint8]
FloatArray = NDArray[np.float32]

project_root: Path = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

FACE_DIR: Path = project_root / "data" / "face"
MODELS_DIR: Path = project_root / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CBIR_INDEX_PATH: Path = MODELS_DIR / "cbir_method2_index.npz"
CBIR_META_PATH: Path = MODELS_DIR / "cbir_method2_meta.json"

OUTPUT_SIZE: tuple[int, int] = (128, 128)
IMAGE_EXTS: set[str] = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Face folder: {FACE_DIR}")
print(f"CBIR method2 index output: {CBIR_INDEX_PATH}")
print(f"CBIR method2 meta output : {CBIR_META_PATH}")

Face folder: c:\Users\harry\Documents\school\ip\AttSystem\data\face
LBPH output: c:\Users\harry\Documents\school\ip\AttSystem\models\lbph.yml
Label map : c:\Users\harry\Documents\school\ip\AttSystem\models\label_map_lbph.txt


In [ ]:
haar_root = Path(getattr(cv2, "data").haarcascades)  # type: ignore[attr-defined]
face_cascade = cv2.CascadeClassifier(str(haar_root / "haarcascade_frontalface_default.xml"))
if face_cascade.empty():
    raise RuntimeError("Failed to load OpenCV frontal face cascade.")


def detect_face_roi(gray: ImageArray) -> ImageArray:
    h, w = gray.shape[:2]
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(40, 40),
    )

    if len(faces) == 0:
        side: int = int(min(h, w) * 0.75)
        cx, cy = w // 2, h // 2
        x1: int = max(0, cx - side // 2)
        y1: int = max(0, cy - side // 2)
        x2: int = min(w, x1 + side)
        y2: int = min(h, y1 + side)
        return gray[y1:y2, x1:x2].astype(np.uint8, copy=False)

    x, y, fw, fh = max(faces, key=lambda f: int(f[2]) * int(f[3]))
    pad_w: int = int(fw * 0.2)
    pad_h: int = int(fh * 0.2)

    x1: int = max(0, int(x) - pad_w)
    y1: int = max(0, int(y) - pad_h)
    x2: int = min(w, int(x) + int(fw) + pad_w)
    y2: int = min(h, int(y) + int(fh) + pad_h)

    return gray[y1:y2, x1:x2].astype(np.uint8, copy=False)


def preprocess_roi_method2(roi_gray: ImageArray) -> ImageArray:
    equalized = cv2.equalizeHist(roi_gray)
    denoised = cv2.bilateralFilter(equalized, d=7, sigmaColor=50, sigmaSpace=50)
    sharpened = cv2.addWeighted(denoised, 1.35, cv2.GaussianBlur(denoised, (0, 0), 1.2), -0.35, 0)
    resized = cv2.resize(sharpened, OUTPUT_SIZE, interpolation=cv2.INTER_CUBIC)
    return resized.astype(np.uint8, copy=False)


def extract_embedding_from_face(face_gray: ImageArray) -> FloatArray | None:
    face_rgb = cv2.cvtColor(face_gray, cv2.COLOR_GRAY2RGB)
    encodings: list[Any] = face_recognition.face_encodings(face_rgb)
    if len(encodings) == 0:
        return None

    emb = np.array(encodings[0], dtype=np.float32)
    norm = np.linalg.norm(emb)
    if norm <= 1e-12:
        return None
    return (emb / norm).astype(np.float32)


def build_cbir_gallery_method2(face_dir: Path) -> tuple[FloatArray, NDArray[np.int32], list[str], list[str]]:
    if not face_dir.exists() or not face_dir.is_dir():
        raise FileNotFoundError(f"Missing dataset folder: {face_dir}")

    person_dirs: list[Path] = sorted([p for p in face_dir.iterdir() if p.is_dir()])
    if not person_dirs:
        raise RuntimeError("No person folders found in data/face.")

    embeddings: list[FloatArray] = []
    label_ids: list[int] = []
    label_names: list[str] = []
    image_paths: list[str] = []

    total_images: int = 0
    kept_images: int = 0

    for label_id, person_dir in enumerate(person_dirs):
        files: list[Path] = sorted([f for f in person_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])
        if not files:
            continue

        label_names.append(person_dir.name)
        person_kept: int = 0

        for f in files:
            total_images += 1
            bgr = cv2.imread(str(f), cv2.IMREAD_COLOR)
            if bgr is None:
                continue

            gray = cv2.cvtColor(bgr.astype(np.uint8, copy=False), cv2.COLOR_BGR2GRAY).astype(np.uint8, copy=False)
            roi = detect_face_roi(gray)
            if roi.size == 0:
                continue

            sample = preprocess_roi_method2(roi)
            emb = extract_embedding_from_face(sample)
            if emb is None:
                continue

            embeddings.append(emb)
            label_ids.append(label_id)
            image_paths.append(str(f))
            person_kept += 1
            kept_images += 1

        print(f"{person_dir.name}: indexed {person_kept}/{len(files)} images")

    if len(embeddings) == 0:
        raise RuntimeError("No usable face embeddings were produced from data/face.")

    print(f"Total read: {total_images}, indexed: {kept_images}")

    emb_np: FloatArray = np.vstack(embeddings).astype(np.float32)
    label_np: NDArray[np.int32] = np.array(label_ids, dtype=np.int32)
    return emb_np, label_np, label_names, image_paths

In [ ]:
def save_cbir_method2_index() -> None:
    embeddings, labels, label_names, image_paths = build_cbir_gallery_method2(FACE_DIR)

    np.savez_compressed(
        CBIR_INDEX_PATH,
        embeddings=embeddings,
        labels=labels,
        image_paths=np.array(image_paths, dtype=object),
    )

    meta = {
        "embedding_dim": int(embeddings.shape[1]),
        "num_vectors": int(embeddings.shape[0]),
        "label_names": label_names,
        "distance": "cosine_on_l2_normalized",
        "source_folder": str(FACE_DIR),
        "preprocess_method": "method2",
    }

    with open(CBIR_META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print("\nCBIR Method 2 pipeline complete")
    print(f"Index saved: {CBIR_INDEX_PATH}")
    print(f"Meta saved : {CBIR_META_PATH}")
    print(f"Vectors    : {embeddings.shape[0]}")
    print(f"Dim        : {embeddings.shape[1]}")


save_cbir_method2_index()

benjamin: used 1080/1080 images
chern_tak: used 1008/1008 images
chillien: used 324/324 images
daniel: used 1080/1080 images
dylan: used 756/756 images
han_soon: used 720/720 images
harry: used 360/360 images
isaac: used 360/360 images
jing_ang: used 1080/1080 images
jun_wei: used 972/972 images
kang_kai: used 1080/1080 images
marion: used 1008/1008 images
ms_nurul: used 396/396 images
qi_xuan: used 1080/1080 images
shuang_quan: used 1620/1620 images
wee_xuan: used 864/864 images
xiang_yue: used 1080/1080 images
xu_sheng: used 1044/1044 images
yoke_hong: used 1152/1152 images
yong_kang: used 864/864 images
zi_herng: used 360/360 images
Total read: 18288, usable: 18288

Pipeline 2 (LBPH) complete
Model saved: c:\Users\harry\Documents\school\ip\AttSystem\models\lbph.yml
Label map : c:\Users\harry\Documents\school\ip\AttSystem\models\label_map_lbph.txt
Samples   : 18288
People    : 21
